# Autonomous News Reporter
Analyses the current news by region or topic, summerises it and sends a notification depending on importance and region affected.



In [ ]:
import os
from dotenv import load_dotenv
from agents import Agent, Runner, trace, Tool
from agents.mcp import MCPServerStdio
from IPython.display import Markdown, display
from datetime import datetime

load_dotenv(override=True)
brave_env = {"BRAVE_API_KEY": os.getenv("BRAVE_API_KEY")}

### World News MCP Server
MCP server to gather current world news from different sources.

In [ ]:
# News MCP tools
display(Markdown(f"### News MCP tools"))
async with MCPServerStdio(params={"command": "npx", "args": ["-y", "@newsmcp/server"]}, client_session_timeout_seconds=60) as server:
    mcp_tools = await server.list_tools()
for tool in mcp_tools:
    print(f"{tool.name} - {tool.description}")

# Fetch News MCP tools
display(Markdown(f"### Browser MCP tools"))
async with MCPServerStdio(params={"command": "npx", "args": ["-y", "@modelcontextprotocol/server-brave-search"], "env": brave_env}, client_session_timeout_seconds=60) as server:
    mcp_tools = await server.list_tools()
for tool in mcp_tools:
    print(f"{tool.name} - {tool.description}")



## News Repoter

In [ ]:
NEWS_REPORTER_MODEL = "gpt-4o-mini"

world_news_mcp_params = [
    {"command": "npx", "args": ["-y", "@modelcontextprotocol/server-brave-search"], "env": brave_env},
    {"command": "npx", "args": ["-y", "@newsmcp/server"]}
]

world_news_mcp_servers = [MCPServerStdio(params) for params in world_news_mcp_params]

# News Reporter Agent
async def get_news_reporter(mcp_servers) -> Agent:
    instructions = f"""
    You are a news reporter. You are able to get the latest news about interesting topics or regions from the world.
    Search for at most 9 news articles and summarize them in 2-3 paragraphs. Concetrate mostly on the latest
    news and the most important details.
    Please take your time to make multiple searches to get a comprehensive overview, and then summarize your findings.
    Also, please verify the sources of the news to ensure the accuracy and credibility of the information.
    """
    news_reporter = Agent(
        name="News Reporter",
        instructions=instructions,
        model=NEWS_REPORTER_MODEL,
        mcp_servers=mcp_servers
    )
    return news_reporter

In [ ]:
async def get_news_reporter_tool(mcp_servers) -> Tool:
    news_reporter = await get_news_reporter(mcp_servers)
    return news_reporter.as_tool(
        tool_name="news_reporter",
        tool_description="Get the latest news about interesting topics or regions from the world.",
    )

    


In [ ]:
news_question = "What's the latest breaking news in Africa?"

for server in world_news_mcp_servers:
    await server.connect()
    
news_reporter = await get_news_reporter(world_news_mcp_servers)
with trace("News Reporter"):
    result = await Runner.run(news_reporter, news_question, max_turns=30)
display(Markdown(result.final_output))

## News Broadcaster

In [ ]:
NEWS_BROADCASTER_MODEL = "gpt-4.1-mini"

agent_name = "Africa News"

from pathlib import Path


def _push_server_dir() -> Path:
    """Directory with `push_server.py` (stdio cwd for `uv run`)."""
    cwd = Path.cwd()
    if (cwd / "push_server.py").exists():
        return cwd
    for parent in [cwd, *cwd.parents]:
        candidate = parent / "push_server.py"
        if candidate.exists():
            return candidate.parent
    raise FileNotFoundError(
        "Could not find 6_mcp"
        "Open/run the notebook from the agents repo or from that folder."
    )


_MCP_CWD = _push_server_dir()

print(_MCP_CWD)

# Using MCP Servers to read resources (`cwd` so `uv run` finds scripts; push_server.py must live here)
news_broadcaster_mcp_params = [
    {"command": "uv", "args": ["run", "news_articles_mcp_server.py" ]},
    {"command": "uv", "args": ["run", "push_server.py"], "cwd": _MCP_CWD},
]

news_broadcaster_mcp_servers = [MCPServerStdio(params) for params in news_broadcaster_mcp_params]



In [ ]:
# News Articles MCP tools
display(Markdown(f"### News Articles MCP tools"))
async with MCPServerStdio(params={"command": "uv", "args": ["run", "news_articles_mcp_server.py" ]}, client_session_timeout_seconds=60) as server:
    mcp_tools = await server.list_tools()
for tool in mcp_tools:
    print(f"{tool.name} - {tool.description}")

# Fetch News MCP tools
display(Markdown(f"### Push MCP tools"))
async with MCPServerStdio(params={"command": "uv", "args": ["run", "push_server.py"], "cwd": _MCP_CWD}, client_session_timeout_seconds=60) as server:
    mcp_tools = await server.list_tools()
for tool in mcp_tools:
    print(f"{tool.name} - {tool.description}")

In [ ]:

instructions = f"""
You are a news broadcaster. You are able to broadcast news about interesting topics or regions from the world.
You have access to tools that allow you to browse available news and surf the web for verification. Please save these 
to the database.
You also have access to tools that allow you to send notifications to users for breaking news. Only send notifications for breaking news
or as requested by the user.
Please make use of these tools to save the news. Save news as you see fit; do not wait for instructions or ask for confirmation.
"""

# call `connect()` first to initialize the mcp servers (world news MCP is used by the nested news reporter tool)
for server in world_news_mcp_servers:
    await server.connect()
for server in news_broadcaster_mcp_servers:
    await server.connect()

news_reporter_tool = await get_news_reporter_tool(world_news_mcp_servers)

news_broadcaster = Agent(
    name=agent_name,
    instructions=instructions,
    tools=[news_reporter_tool],
    mcp_servers=news_broadcaster_mcp_servers,
    model=NEWS_BROADCASTER_MODEL,
)




### Chat with the broadcaster

In [ ]:
import gradio as gr

async def broadcaster_chat(message: str, history: list) -> str:
    """Multi-turn chat with the news broadcaster (Gradio `type='messages'`)."""
    try:
        _ = news_broadcaster
    except NameError:
        return "Run the **News Broadcaster** setup cell first so `news_broadcaster` is defined."

    processed = [{"role": m["role"], "content": m["content"]} for m in history]
    current_input = processed + [{"role": "user", "content": message}]
    try:
        with trace(agent_name):
            result = await Runner.run(
                news_broadcaster,
                input=current_input,
                max_turns=30,
            )
        return result.final_output
    except Exception as e:
        return f"**Error:** {e}"


broadcaster_chat_ui = gr.ChatInterface(
    fn=broadcaster_chat,
    type="messages",
    title=f"{agent_name} — Broadcaster",
    description=(
        "Ask for news by region or topic, request saves to the database, or ask for push notifications. "
        "The agent keeps context within this chat session."
    ),
    examples=[
        "What are the top stories in Africa right now? Save the most important article to the database.",
        "Send me a push notification only if there is breaking political news.",
        "Summarize technology headlines and update the saved article with id 1 if needed.",
    ],
    theme=gr.themes.Soft(primary_hue="red"),
)

broadcaster_chat_ui.launch(inline=True)
